# Week 4 - Mini Project: RAG Chatbot on Requests Documentation

This notebook builds a RAG chatbot over the official Requests Python library documentation. It fetches selected official pages, saves them as local text files, chunks them, stores embeddings in ChromaDB, asks technical questions, and computes simple retrieval evaluation metrics.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

from src.rag.python_library_docs import fetch_requests_docs, docs_to_documents
from src.rag.text_splitter import split_documents
from src.rag.vector_store import create_or_load_vector_store
from src.rag.rag_chain import TechStoreRAGAssistant
from src.rag.evaluation import retrieval_metrics, roc_curve_points

load_dotenv()

DOCS_DIR = Path("docs/python_library_docs/requests")
PERSIST_DIR = Path("chroma_db/requests_docs")
COLLECTION = "requests_docs"

## 1. Fetch Official Requests Documentation

In [ ]:
saved_paths = fetch_requests_docs(DOCS_DIR)
saved_paths

## 2. Ingest And Chunk Content

In [ ]:
documents = docs_to_documents(DOCS_DIR)
chunks = split_documents(documents, chunk_size=900, chunk_overlap=160)
len(documents), len(chunks), chunks[0].metadata

## 3. Store In ChromaDB

In [ ]:
vector_store = create_or_load_vector_store(
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION,
)
vector_store.add_documents(chunks)
print(f"Stored {len(chunks)} chunks in {PERSIST_DIR}")

## 4. Q&A Chatbot

In [ ]:
reloaded_store = create_or_load_vector_store(
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION,
)
assistant = TechStoreRAGAssistant(
    retriever=reloaded_store,
    system_prompt="You are a Requests Python library documentation assistant.",
    not_found_message="I could not find that answer in the Requests documentation.",
)

questions = [
    "How do I pass query parameters with requests?",
    "How do I set a timeout?",
    "How do sessions persist cookies?",
    "How do I upload a file?",
    "How should I handle HTTP errors?",
]

for question in questions:
    print("=" * 80)
    print(question)
    print(assistant.answer(question))

## 5. Bonus: Retrieval Evaluation

In [ ]:
evaluation_cases = [
    {
        "question": "How do I pass query parameters with requests?",
        "relevant_ids": {"requests_quickstart.txt"},
    },
    {
        "question": "How do sessions persist cookies?",
        "relevant_ids": {"requests_advanced.txt", "requests_api.txt"},
    },
    {
        "question": "How do I set a request timeout?",
        "relevant_ids": {"requests_quickstart.txt", "requests_advanced.txt"},
    },
]

metric_cases = []
scored_results = []
for case in evaluation_cases:
    docs = reloaded_store.similarity_search(case["question"], k=4)
    retrieved_ids = [Path(doc.metadata.get("source", "unknown")).name for doc in docs]
    metric_cases.append({**case, "retrieved_ids": retrieved_ids})
    for rank, doc in enumerate(docs, start=1):
        doc_id = Path(doc.metadata.get("source", "unknown")).name
        scored_results.append({
            "score": 1 / rank,
            "relevant": doc_id in case["relevant_ids"],
        })

retrieval_metrics(metric_cases), roc_curve_points(scored_results)